### Model Trainer 

In [1]:
import os

In [2]:
# Set this to  actual project path
project_path = r"c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline"

os.chdir(project_path)

print(f"Now at: {os.getcwd()}")

Now at: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline


In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [4]:
from src.TextInsightMlopsPipeline.constants import *
from src.TextInsightMlopsPipeline.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )

        return model_trainer_config

In [6]:
pip install torch
# Ctrl + Shift + P → Restart Kernel

SyntaxError: invalid syntax (3304121106.py, line 1)

In [7]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            do_eval=True,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

In [9]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.4.0-py3-none-any.whl (10.1 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# config = ConfigurationManager()
# model_trainer_config = config.get_model_trainer_config()
# model_trainer = ModelTrainer(config=model_trainer_config)
# model_trainer.train()
# This will take around 1 hour to train on GPU and around 4-5 hours on CPU.
# So I trained the model on Google Colab and saved the trained model in Google Drive. You can download the trained model from the following link:

# Pegasus Samsum Model->  https://drive.google.com/drive/folders/1LaT8NLPKlBVg-0AErwL-MSdKSULUNgzP?usp=sharing
# Tokenizer-> https://drive.google.com/drive/folders/1R9FoH6k11Dzm4u3O7zACzdn1pi07iMvO?usp=sharing

In [5]:
!pip install gdown -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import gdown
import os

os.makedirs("artifacts/model_trainer/pegasus-samsum-model", exist_ok=True)
os.makedirs("artifacts/model_trainer/tokenizer", exist_ok=True)

gdown.download_folder(
    "https://drive.google.com/drive/folders/1LaT8NLPKlBVg-0AErwL-MSdKSULUNgzP",
    output="artifacts/model_trainer/pegasus-samsum-model",
    quiet=False
)

gdown.download_folder(
    "https://drive.google.com/drive/folders/1R9FoH6k11Dzm4u3O7zACzdn1pi07iMvO",
    output="artifacts/model_trainer/tokenizer",
    quiet=False
)

print("Done!")

Retrieving folder contents


Processing file 1oPEVZE0mbQ7fIjDyTSgYjFucKuW_7gp3 config.json
Processing file 12qA0Q2CZYNDF7l1lCkAM_L5S5s17EUa5 generation_config.json
Processing file 15hCEyNNtLKpuDFMcI3lzo49wSNctcS6z model.safetensors


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1oPEVZE0mbQ7fIjDyTSgYjFucKuW_7gp3
To: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\research\artifacts\model_trainer\pegasus-samsum-model\config.json
100%|██████████| 1.27k/1.27k [00:00<00:00, 1.48MB/s]
Downloading...
From: https://drive.google.com/uc?id=12qA0Q2CZYNDF7l1lCkAM_L5S5s17EUa5
To: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\research\artifacts\model_trainer\pegasus-samsum-model\generation_config.json
100%|██████████| 275/275 [00:00<00:00, 851kB/s]
Downloading...
From (original): https://drive.google.com/uc?id=15hCEyNNtLKpuDFMcI3lzo49wSNctcS6z
From (redirected): https://drive.google.com/uc?id=15hCEyNNtLKpuDFMcI3lzo49wSNctcS6z&confirm=t&uuid=7ac4c32e-d076-4797-8a21-f971bdde973a
To: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\research\artifacts\model_trainer\pegasus-samsum-model\

Processing file 1lEWU7rErvyAnMfBM8LKDp0U7A23tRHz1 special_tokens_map.json
Processing file 1DTmS2mtBJ8-s9mpMp2FXX8dZeLnshSFG spiece.model
Processing file 1rIlsRc0CEylRfqO_us4STVh6ac2aGyBu tokenizer_config.json
Processing file 11GMnFPPNy-4EpluMRDSqHhC9OErV_cvE tokenizer.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1lEWU7rErvyAnMfBM8LKDp0U7A23tRHz1
To: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\research\artifacts\model_trainer\tokenizer\special_tokens_map.json
100%|██████████| 1.77k/1.77k [00:00<00:00, 2.32MB/s]
Downloading...
From: https://drive.google.com/uc?id=1DTmS2mtBJ8-s9mpMp2FXX8dZeLnshSFG
To: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\research\artifacts\model_trainer\tokenizer\spiece.model
100%|██████████| 1.91M/1.91M [00:01<00:00, 1.89MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rIlsRc0CEylRfqO_us4STVh6ac2aGyBu
To: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\research\artifacts\model_trainer\tokenizer\tokenizer_config.json
100%|██████████| 20.3k/20.3k [00:00<00:00, 237kB/s]
Downloading...
From: https://drive.google.com/uc?id=11GMnFPPNy-4EpluMRDSqHhC9OErV_cvE
To: c:\Users\

Done!



Download completed


In [ ]:
import shutil
import os

shutil.move(
    "artifacts/model_trainer",
    r"c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\artifacts\model_trainer"
)

print("Moved successfully!")

Moved successfully!
